In [ ]:
!pip install opendatasets

In [ ]:
import opendatasets as od
od.download("https://www.kaggle.com/datasets/serkantysz/550k-spotify-songs-audio-lyrics-and-genres?select=songs.csv")
od.download("https://www.kaggle.com/datasets/dhruvildave/billboard-the-hot-100-songs")

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: cyang1
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/serkantysz/550k-spotify-songs-audio-lyrics-and-genres


100%|██████████| 235M/235M [00:02<00:00, 95.1MB/s]



Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: cyang1
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/dhruvildave/billboard-the-hot-100-songs


100%|██████████| 3.05M/3.05M [00:00<00:00, 39.8MB/s]

In [ ]:
import pandas as pd

df = pd.read_csv("/content/550k-spotify-songs-audio-lyrics-and-genres/songs.csv")

df['decade'] = (df['year'] // 10 * 10).astype(str) + 's'

decade_counts = df.groupby('decade').size()

for decade, count in decade_counts.items():
    print(f"{decade}: {count:,} songs")

1900s: 30 songs
1910s: 3 songs
1920s: 156 songs
1930s: 288 songs
1940s: 437 songs
1950s: 4,119 songs
1960s: 11,344 songs
1970s: 17,570 songs
1980s: 20,729 songs
1990s: 42,913 songs
2000s: 162,161 songs
2010s: 225,362 songs
2020s: 65,510 songs


In [ ]:
artists_df = pd.read_csv("/content/550k-spotify-songs-audio-lyrics-and-genres/artists.csv")

print(artists_df.head())
print(artists_df.shape)
print(artists_df.columns.tolist())

                       id               name  followers  popularity  \
0  6YROFUbu5zRCHi2xkir5pk       Brian Hyland      67223          47   
1  5tFRohaO5yEsuJxmMnlCO9     Barns Courtney     602647          62   
2  3w1Q754jb31h5CXQCcnLNL  Capcom Sound Team     210392          58   
3  3oDbviiivRWhXwIE8hxkVV     The Beach Boys    5139194          76   
4  60zvRmhQHRxokEB1taAVpN        Beth Malone       1569          29   

                           genres  main_genre  
0                              []         Pop  
1                              []  Electronic  
2  ['japanese vgm', 'soundtrack']  Electronic  
3                 ['baroque pop']   Classical  
4                    ['musicals']   Classical  
(71440, 6)
['id', 'name', 'followers', 'popularity', 'genres', 'main_genre']


In [ ]:
# downsizing to match 1960s

def balance_decades(df, start_decade='1960s', seed=42):
    # find the target count from the start decade
    target_count = len(df[df['decade'] == start_decade])
    print(f"Target count (based on {start_decade}): {target_count:,}")

    balanced_decades = []

    for decade, group in df.groupby('decade'):
        # skip
        if int(decade[:4]) < 1960:
            continue

        if len(group) >= target_count:
            sampled = group.sample(n=target_count, random_state=seed)
        else:
            print(f"Warning: {decade} only has {len(group):,} songs, keeping all.")
            sampled = group

        balanced_decades.append(sampled)

    return pd.concat(balanced_decades).reset_index(drop=True)

balanced_df = balance_decades(df)

print("\nSong counts per decade after balancing:")
for decade, count in balanced_df.groupby('decade').size().items():
    print(f"  {decade}: {count:,} songs")

Target count (based on 1960s): 11,344

Song counts per decade after balancing:
  1960s: 11,344 songs
  1970s: 11,344 songs
  1980s: 11,344 songs
  1990s: 11,344 songs
  2000s: 11,344 songs
  2010s: 11,344 songs
  2020s: 11,344 songs


In [ ]:
charts_df = pd.read_csv("/content/billboard-the-hot-100-songs/charts.csv")

charts_df['decade'] = (pd.to_datetime(charts_df['date']).dt.year // 10 * 10).astype(str) + 's'

def get_artists_by_rank_threshold(df, threshold=100, start_decade='1960s'):
    filtered = df[df['rank'] <= threshold]

    results = {}

    for decade, group in filtered.groupby('decade'):
        if int(decade[:4]) < 1960:
            continue

        unique_artists = group['artist'].unique().tolist()
        results[decade] = unique_artists

    return results


def compare_thresholds(df, start_decade='1960s'):
    thresholds = [100, 50, 10]

    all_results = {}
    for t in thresholds:
        all_results[t] = get_artists_by_rank_threshold(df, threshold=t, start_decade=start_decade)

    for decade in all_results[100].keys():
        print(f"\n{'='*40}")
        print(f" {decade}")
        print(f"{'='*40}")
        for t in thresholds:
            artists = all_results[t].get(decade, [])
            print(f"  Top {t:>3}: {len(artists):>4} unique artists")
            print(f"          Sample: {artists[:3]}")

    return all_results


all_results = compare_thresholds(charts_df)


 1960s
  Top 100: 1982 unique artists
          Sample: ['Diana Ross & The Supremes', 'Peter, Paul & Mary', 'B.J. Thomas']
  Top  50: 1176 unique artists
          Sample: ['Diana Ross & The Supremes', 'Peter, Paul & Mary', 'B.J. Thomas']
  Top  10:  540 unique artists
          Sample: ['Diana Ross & The Supremes', 'Peter, Paul & Mary', 'B.J. Thomas']

 1970s
  Top 100: 1837 unique artists
          Sample: ['Rupert Holmes', 'KC And The Sunshine Band', 'Styx']
  Top  50: 1101 unique artists
          Sample: ['Rupert Holmes', 'KC And The Sunshine Band', 'Styx']
  Top  10:  504 unique artists
          Sample: ['Rupert Holmes', 'KC And The Sunshine Band', 'Styx']

 1980s
  Top 100: 1541 unique artists
          Sample: ['Phil Collins', 'Linda Ronstadt (Featuring Aaron Neville)', 'Billy Joel']
  Top  50:  965 unique artists
          Sample: ['Phil Collins', 'Linda Ronstadt (Featuring Aaron Neville)', 'Billy Joel']
  Top  10:  443 unique artists
          Sample: ['Phil Collins', 'Lind

In [ ]:
# cleaning artist name
def clean_artist_name(artist):
    """Strip out featured artists so 'Drake Featuring Future' becomes 'Drake'"""
    for separator in [' Featuring ', ' feat. ', ' Feat. ', ' ft. ', ' Ft. ', ' x ', ' X ', ' & ', ' With ', ' with ']:
        artist = artist.split(separator)[0]
    return artist.strip()

def get_artists_by_rank_threshold(df, threshold=100, start_decade='1960s'):
    filtered = df[df['rank'] <= threshold].copy()

    filtered['main_artist'] = filtered['artist'].apply(clean_artist_name)

    results = {}

    for decade, group in filtered.groupby('decade'):
        if int(decade[:4]) < 1960:
            continue

        unique_artists = sorted(group['main_artist'].unique().tolist())
        results[decade] = unique_artists

    return results


def compare_thresholds(df, start_decade='1960s'):
    thresholds = [100, 50, 10]

    all_results = {}
    for t in thresholds:
        all_results[t] = get_artists_by_rank_threshold(df, threshold=t, start_decade=start_decade)

    for decade in all_results[100].keys():
        print(f"\n{'='*40}")
        print(f" {decade}")
        print(f"{'='*40}")
        for t in thresholds:
            artists = all_results[t].get(decade, [])
            print(f"  Top {t:>3}: {len(artists):>4} unique artists")
            print(f"          Sample: {artists[:5]}")

    return all_results


all_results = compare_thresholds(charts_df)


 1960s
  Top 100: 1900 unique artists
          Sample: ['"Groove" Holmes', '"Little" Jimmy Dickens', '"Pookie" Hudson', '100 Proof Aged in Soul', '100 Strings and Jono (Choir of 40 Voices)']
  Top  50: 1137 unique artists
          Sample: ['"Groove" Holmes', '"Little" Jimmy Dickens', '1910 Fruitgum Co.', '? (Question Mark)', 'Aaron Neville']
  Top  10:  529 unique artists
          Sample: ['1910 Fruitgum Co.', '? (Question Mark)', 'Aaron Neville', 'Adam Wade', "Al (He's the King) Hirt"]

 1970s
  Top 100: 1760 unique artists
          Sample: ['(The Preacher) Bobby Womack', '100 Proof Aged in Soul', '10cc', '5000 Volts', 'A Taste Of Honey']
  Top  50: 1055 unique artists
          Sample: ['100 Proof Aged in Soul', '10cc', '5000 Volts', 'A Taste Of Honey', 'ABBA']
  Top  10:  486 unique artists
          Sample: ['100 Proof Aged in Soul', '10cc', 'A Taste Of Honey', 'ABBA', 'AWB']

 1980s
  Top 100: 1452 unique artists
          Sample: ['"Weird Al" Yankovic', "'Til Tuesday", '10,0

In [ ]:
import re

#combining artists/songs/rankings 
songs_df = pd.read_csv('/content/550k-spotify-songs-audio-lyrics-and-genres/songs.csv')
artists_df = pd.read_csv('/content/550k-spotify-songs-audio-lyrics-and-genres/artists.csv')
charts_df = pd.read_csv('/content/billboard-the-hot-100-songs/charts.csv')

def normalize_text(text):
    if pd.isna(text):
        return ''
    text = text.lower().strip()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text

def clean_artist_name(artist):
    for separator in [' featuring ', ' feat. ', ' feat ', ' ft. ', ' ft ', ' x ', ' & ', ' with ']:
        artist = artist.lower().split(separator)[0]
    return artist.strip()

# normalize artists_df 
artists_df['name_normalized'] = artists_df['name'].apply(normalize_text)

# normalize charts_df artist column
charts_df['artist_normalized'] = charts_df['artist'].apply(
    lambda x: clean_artist_name(normalize_text(x))
)
charts_df['song_normalized'] = charts_df['song'].apply(normalize_text)

# normalize songs_df
songs_df['name_normalized'] = songs_df['name'].apply(normalize_text)

artist_chart_merge = pd.merge(
    artists_df,
    charts_df,
    left_on='name_normalized',
    right_on='artist_normalized',
    how='inner'
)

print(f"Artists in Spotify artists.csv:     {len(artists_df):,}")
print(f"Entries in Billboard charts.csv:    {len(charts_df):,}")
print(f"Matched on artist name:             {len(artist_chart_merge):,}")
print(f"Unique matched artists:             {artist_chart_merge['name'].nunique():,}")

# step 2: now also bring in the songs_df to confirm song name matches
full_merge = pd.merge(
    artist_chart_merge,
    songs_df[['name_normalized', 'genre', 'year']],
    left_on='song_normalized',
    right_on='name_normalized',
    how='inner'
)

print(f"\nAfter also matching song names:     {len(full_merge):,}")
print(f"Unique matched songs:               {full_merge['song_normalized'].nunique():,}")

print("\nAvailable columns:")
print(full_merge.columns.tolist())

# sleecting columns of what we need
final_df = full_merge[['name', 'song', 'artist', 'year', 'genre', 'rank', 'date']].copy()
final_df = final_df.rename(columns={'name': 'artist_name', 'song': 'song_name'})
final_df = final_df.drop_duplicates(subset=['song_name', 'artist_name', 'year'])

print("\nSample matched records:")
print(final_df.head(10))

Artists in Spotify artists.csv:     71,440
Entries in Billboard charts.csv:    330,087
Matched on artist name:             278,360
Unique matched artists:             4,213

After also matching song names:     2,675,734
Unique matched songs:               13,641

Available columns:
['id', 'name', 'followers', 'popularity', 'genres', 'main_genre', 'name_normalized_x', 'date', 'rank', 'song', 'artist', 'last-week', 'peak-rank', 'weeks-on-board', 'artist_normalized', 'song_normalized', 'name_normalized_y', 'genre', 'year']

Sample matched records:
     artist_name         song_name        artist  year    genre  rank  \
0   Brian Hyland  Lonely Teardrops  Brian Hyland  2006      Pop    67   
1   Brian Hyland  Lonely Teardrops  Brian Hyland  1979     Rock    67   
2   Brian Hyland  Lonely Teardrops  Brian Hyland  1959      Pop    67   
3   Brian Hyland  Lonely Teardrops  Brian Hyland  2002      Pop    67   
4   Brian Hyland  Lonely Teardrops  Brian Hyland  2009  Country    67   
40  Brian H

In [ ]:
def get_top10_per_decade(df):
    df['decade'] = (pd.to_datetime(df['date']).dt.year // 10 * 10).astype(str) + 's'

    results = {}

    for decade, group in df.groupby('decade'):
        if int(decade[:4]) < 1960:
            continue

        artist_stats = group.groupby('artist_name').agg(
            total_chart_appearances = ('rank', 'count'),
            best_rank               = ('rank', 'min'),
            avg_rank                = ('rank', 'mean'),
            unique_songs            = ('song_name', 'nunique'),
            times_at_number_1       = ('rank', lambda x: (x == 1).sum())
        ).reset_index()

        # sort by most chart appearances, then best rank as tiebreaker
        artist_stats = artist_stats.sort_values(
            by=['total_chart_appearances', 'best_rank'],
            ascending=[False, True]
        ).head(10)

        artist_stats['decade'] = decade
        artist_stats['chart_rank'] = range(1, len(artist_stats) + 1)
        results[decade] = artist_stats

    return results


top10 = get_top10_per_decade(final_df)

for decade, df_decade in top10.items():
    print(f"\n{'='*60}")
    print(f"  Top 10 Artists — {decade}")
    print(f"{'='*60}")
    print(f"{'#':<4} {'Artist':<30} {'Appearances':<14} {'Best Rank':<12} {'#1 Hits':<10} {'Songs'}")
    print(f"{'-'*60}")
    for _, row in df_decade.iterrows():
        print(f"{int(row['chart_rank']):<4} {row['artist_name']:<30} {int(row['total_chart_appearances']):<14} {int(row['best_rank']):<12} {int(row['times_at_number_1']):<10} {int(row['unique_songs'])}")


  Top 10 Artists — 1960s
#    Artist                         Appearances    Best Rank    #1 Hits    Songs
------------------------------------------------------------
1    Elvis Presley                  452            26           0          63
2    The Beatles                    413            3            0          45
3    Ray Charles                    369            30           0          34
4    Aretha Franklin                362            26           0          30
5    Otis Redding                   234            35           0          25
6    Sam Cooke                      232            25           0          29
7    The Drifters                   212            35           0          22
8    Frank Sinatra                  206            32           0          24
9    The Beach Boys                 201            21           0          31
10   Johnny Tillotson               195            41           0          18

  Top 10 Artists — 1970s
#    Artist               

In [ ]:
def get_top10_per_decade_with_songs(df):
    df['decade'] = (pd.to_datetime(df['date']).dt.year // 10 * 10).astype(str) + 's'

    results = {}

    for decade, group in df.groupby('decade'):
        if int(decade[:4]) < 1960:
            continue

        artist_stats = group.groupby('artist_name').agg(
            total_chart_appearances = ('rank', 'count'),
            best_rank               = ('rank', 'min'),
            avg_rank                = ('rank', 'mean'),
            unique_songs            = ('song_name', 'nunique'),
            times_at_number_1       = ('rank', lambda x: (x == 1).sum())
        ).reset_index()

        artist_stats = artist_stats.sort_values(
            by=['total_chart_appearances', 'best_rank'],
            ascending=[False, True]
        ).head(10)

        artist_stats['decade'] = decade
        artist_stats['chart_rank'] = range(1, len(artist_stats) + 1)

        # get the top 5 songs for each artist in this decade
        # ranked by their best chart position
        def get_top_songs(artist):
            artist_songs = group[group['artist_name'] == artist].groupby('song_name').agg(
                best_rank        = ('rank', 'min'),
                weeks_on_chart   = ('rank', 'count'),
                times_at_number_1 = ('rank', lambda x: (x == 1).sum())
            ).reset_index()

            artist_songs = artist_songs.sort_values(
                by=['best_rank', 'weeks_on_chart'],
                ascending=[True, False]
            ).head(5)

            return artist_songs[['song_name', 'best_rank', 'weeks_on_chart', 'times_at_number_1']].to_dict('records')

        artist_stats['top_songs'] = artist_stats['artist_name'].apply(get_top_songs)
        results[decade] = artist_stats

    return results


top10_with_songs = get_top10_per_decade_with_songs(final_df)

for decade, df_decade in top10_with_songs.items():
    print(f"\n{'='*70}")
    print(f"  Top 10 Artists — {decade}")
    print(f"{'='*70}")

    for _, row in df_decade.iterrows():
        print(f"\n  #{int(row['chart_rank'])}  {row['artist_name']}")
        print(f"      Chart appearances: {int(row['total_chart_appearances']):,}  |  "
              f"Best rank: #{int(row['best_rank'])}  |  "
              f"#1 hits: {int(row['times_at_number_1'])}  |  "
              f"Unique songs: {int(row['unique_songs'])}")
        print(f"      Top songs:")
        for song in row['top_songs']:
            number1 = ' ← #1 hit' if song['times_at_number_1'] > 0 else ''
            print(f"        • {song['song_name']:<40} "
                  f"peak: #{song['best_rank']:<5} "
                  f"weeks: {song['weeks_on_chart']}{number1}")


  Top 10 Artists — 1960s

  #1  Elvis Presley
      Chart appearances: 452  |  Best rank: #26  |  #1 hits: 0  |  Unique songs: 63
      Top songs:
        • In The Ghetto                            peak: #26    weeks: 9
        • Frankie And Johnny                       peak: #28    weeks: 6
        • She's Not You                            peak: #32    weeks: 6
        • Such A Night                             peak: #33    weeks: 7
        • Do The Clam                              peak: #34    weeks: 2

  #2  The Beatles
      Chart appearances: 413  |  Best rank: #3  |  #1 hits: 0  |  Unique songs: 45
      Top songs:
        • Something                                peak: #3     weeks: 23
        • I Want To Hold Your Hand                 peak: #24    weeks: 8
        • Love Me Do                               peak: #24    weeks: 3
        • Penny Lane                               peak: #25    weeks: 2
        • Get Back                                 peak: #27    weeks: 17

